In [ ]:
from google.colab import drive, userdata
import json, os, pathlib, subprocess
drive.mount("/content/drive")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
DRIVE = pathlib.Path("/content/drive/MyDrive/msml612")
DATASET = DRIVE / "physionet_set_a.npz"
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
!git clone -q https://github.com/jkelle11-source/msml612_project.git
%cd /content/msml612_project
started = DRIVE / "dry-run" / "sweep.json"
if started.exists():        # a later session: the commit the dry run started from
    subprocess.run(["git", "checkout", "-q", json.loads(started.read_text())["commit"]], check=True)
!git log --oneline -1
!uv sync --locked
!uv run python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!sha256sum {DATASET}

In [ ]:
!uv run python -m diffusion.sweep --tag rerun --epochs 20 --regime random --variants base --seeds 42 --out {DRIVE}/rerun --dataset {DATASET}
!uv run python -m diffusion.sweep --regime random --variants base --seeds 42 --out {DRIVE}/dry-run --compare {DRIVE}/rerun